# EDA — Saúde e falhas de baterias de veículos elétricos

Adaptação do notebook da Aula 11 de Análise e Visualização de Dados.
Fonte: dataset Kaggle `srisyra02/ev-battery-health-prediction-dataset-20k`.

**Perguntas:** como saúde da bateria, resistência interna, idade e ciclos se relacionam? As distribuições diferem entre registros com e sem falha? Quanto o tratamento de ausentes altera a amostra?

O notebook preserva a sequência estrutural → distribuição → tratamento → relações. As interpretações já escritas pelo aluno são hipóteses de leitura a conferir com as saídas da execução; não representam causalidade nem validação da procedência dos dados.

**Execução local:** ative o `.venv` e instale `pandas numpy matplotlib seaborn scipy kagglehub jupyter ipykernel`. Execute as células em ordem. O CSV local em `data/raw/ev_battery_original.csv` tem prioridade; na primeira execução, se ele não existir, será baixado pelo Kaggle. O dado tratado será exportado para `data/clean/ev_battery_clean.csv`.

Ao abrir este arquivo dentro de `notebooks/`, a raiz do projeto será a pasta acima. Em uma pasta isolada, as pastas `data/` serão criadas no diretório de execução.

## Setup de Bibliotecas e Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

COR_NEUTRA = "lightgray"     # convenção: cinza-claro pra tudo que é "pano de fundo" genérico
COR_CONTEXTO = "#e9d4d0"    # rosa-claro pra pano de fundo / classe de referência
COR_DESTAQUE = "#2d223a"    # roxo pra classe de interesse / "o personagem da história"
COR_ALERTA = "tab:red"       # exceção deliberada: outliers/pontos de atenção

## Carregando o Dataset

In [ ]:
from pathlib import Path

# Compatível com execução a partir da raiz do projeto ou de notebooks/.
PASTA_ATUAL = Path.cwd().resolve()
RAIZ = PASTA_ATUAL.parent if PASTA_ATUAL.name == "notebooks" else PASTA_ATUAL
PASTA_RAW = RAIZ / "data" / "raw"
PASTA_CLEAN = RAIZ / "data" / "clean"
PASTA_RAW.mkdir(parents=True, exist_ok=True)
PASTA_CLEAN.mkdir(parents=True, exist_ok=True)
ARQUIVO_RAW = PASTA_RAW / "ev_battery_original.csv"

if ARQUIVO_RAW.exists():
    df_original = pd.read_csv(ARQUIVO_RAW)
else:
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError("Instale kagglehub no ambiente do notebook ou coloque o CSV em "
                          + str(ARQUIVO_RAW)) from exc
    caminho_download = Path(kagglehub.dataset_download(
        "srisyra02/ev-battery-health-prediction-dataset-20k"))
    arquivo_fonte = caminho_download / "ev battery_failure prediction Dataset.csv"
    if not arquivo_fonte.exists():
        raise FileNotFoundError(f"CSV esperado não encontrado em {caminho_download}")
    df_original = pd.read_csv(arquivo_fonte)
    df_original.to_csv(ARQUIVO_RAW, index=False)

df = df_original.copy(deep=True)
print(f"Dados originais: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"CSV original: {ARQUIVO_RAW}")
df.head()

## 1. Princípio Estrutural

Inspecionar dimensões, tipos, duplicadas e ausentes antes de qualquer transformação. Cada linha é um registro do dataset; a variável alvo é `battery_failure` (0 = sem falha, 1 = com falha). Identificadores como `vehicle_id` e `battery_serial` não entram na correlação.

### Dimensão e Tipos de Dados

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### Duplicadas

In [ ]:
df.duplicated().sum()

### Identificando o alvo e preservando os dados originais

O alvo já é binário. Criamos `falha_label` para apresentação, mantendo `battery_failure` para cálculos. Não há recodificação de variável médica neste dataset.

In [ ]:
if "battery_failure" not in df.columns:
    raise ValueError("O dataset precisa conter battery_failure.")
if df["battery_failure"].isna().any() or not df["battery_failure"].isin([0, 1]).all():
    raise ValueError("battery_failure deve conter apenas 0 e 1, sem ausentes.")
df["falha_label"] = df["battery_failure"].map({0: "Sem falha", 1: "Com falha"})
df["falha_label"].value_counts()

## 2. Dados Faltantes

In [ ]:
df.isna().sum()

**Discussao:** Alguma feature tem dados ausentes nos seus dados?

**Verificação na versão analisada:** são 20.000 registros e 70 colunas. Há ausentes em 67 colunas; `vehicle_id`, `battery_serial` e `battery_failure` estão completos. A tabela de ausências acima deve ser consultada novamente caso a versão do dataset mude.

#### Identificando o mecanismo de dados ausentes

**Conhecimento de domínio e limites da análise**

Sem documentação do processo de coleta, não podemos afirmar se os ausentes são MCAR, MAR ou MNAR. Falhas de cadastro e de sensores são hipóteses, não fatos verificados. Vamos quantificar as ausências e examinar sua coocorrência antes de escolher o tratamento.

**As ausencias ocorrem juntas?**

Contar ausentes por coluna não conta a história toda: se duas colunas ficam ausentes **nas mesmas linhas**, isso é um indício de padrão sistemático (ex: um exame que não foi feito, afetando várias medidas de uma vez).



In [ ]:
cols_com_ausentes = df.columns[df.isna().any()].tolist()
print("Colunas com ausentes:", cols_com_ausentes)
print(df[cols_com_ausentes].isna().sum())


A tabela seguinte conta a coocorrência de ausentes nas duas primeiras colunas que os possuem. É uma inspeção inicial, não uma análise de todas as combinações nem um teste que determine o mecanismo de ausência.

In [ ]:
if len(cols_com_ausentes) >= 2:
    col_a, col_b = cols_com_ausentes[:2]
    tabela_contingencia = pd.crosstab(
        df[col_a].isna(), df[col_b].isna(),
        rownames=[f"{col_a} ausente"], colnames=[f"{col_b} ausente"])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(tabela_contingencia, annot=True, fmt="d", cmap="magma", ax=ax)
    ax.set_title("As ausências ocorrem juntas?")
    plt.tight_layout()
    plt.show()
else:
    print("Há menos de duas colunas com ausentes; comparação não se aplica.")

**Discussão:** quantos registros têm as duas colunas ausentes simultaneamente? Compare a frequência conjunta às frequências marginais. Uma pequena contagem conjunta, isoladamente, não comprova independência nem identifica MCAR/MAR/MNAR.

**Mecanismos de ausência:** MCAR significa ausência independente dos valores observados e não observados; MAR permite dependência de valores observados; MNAR envolve dependência dos próprios valores ausentes, mesmo condicionando nos observados. A inspeção exploratória não é suficiente para escolher entre esses mecanismos.

Com muitas colunas, um mapa de ausências pode complementar a análise. Qualquer hipótese sobre o mecanismo precisa ser confrontada com a documentação da fonte.

## 3. Princípio Distribucional

Ainda com os dados originais (sem tratar ausentes ou outliers), vamos calcular as estatísticas descritivas e olhar as distribuições. Esse retrato "antes" é o que vai *orientar as decisões de tratamento* na próxima seção. Por exemplo, se uma variável for muito assimétrica, faz mais sentido imputar com a mediana do que com a média.

In [ ]:
VARS_QUANT = [
    "vehicle_age_years", "battery_capacity_kwh", "odometer_km",
    "cycle_count", "battery_health_percent", "state_of_health",
    "internal_resistance", "charge_efficiency", "discharge_efficiency",
    "average_ambient_temperature", "cell_temperature_max",
    "aging_score", "predicted_remaining_life_cycles"
]

VARS_QUALI = [
    "vehicle_brand", "vehicle_type", "battery_manufacturer",
    "battery_chemistry", "drive_type", "fleet_or_private",
    "terrain_type", "battery_failure"
]
colunas_faltantes = sorted(set(VARS_QUANT + VARS_QUALI) - set(df.columns))
if colunas_faltantes:
    raise ValueError(f"Colunas esperadas ausentes: {colunas_faltantes}")

### Tabela descritiva, com assimetria e curtose
COR_NEUTRA = "#8073ac"
desc = df[VARS_QUANT].describe().T
desc["skewness"] = df[VARS_QUANT].skew()
desc["kurtosis"] = df[VARS_QUANT].kurtosis()

(
    desc[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    
    
)

**Resultado na versão analisada:** entre as 13 quantitativas selecionadas, nenhuma tem |assimetria| > 1 ou |curtose excedente| > 2. `odometer_km` tem assimetria de aproximadamente 0,85 e `cycle_count`, 0,78, sugerindo caudas à direita. Esses indicadores não comprovam normalidade e os limiares são heurísticos.

### Distribuições de dados

Onde fizer sentido, comparamos "Sem falha" (rosa) vs. "Com falha" (roxo). Vamos utilizar esta convenção de cor que vamos manter até a análise bivariada.


In [ ]:
import math

n = len(VARS_QUANT)
ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), sharey=True)
axes = axes.ravel()

for i, col in enumerate(VARS_QUANT):
    sns.histplot(data=df, x=col, kde=True, color=COR_NEUTRA, ax=axes[i])
    axes[i].set_title(col)

for ax in axes[len(VARS_QUANT):]:
    ax.axis("off")

fig.suptitle("Distribuição das Variáveis Quantitativas: EV Battery Failure (sem separar por classe)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(VARS_QUANT), 2, figsize=(10, 3.5 * len(VARS_QUANT)), sharey=True)

for i, col in enumerate(VARS_QUANT):
    lim = (df[col].min(), df[col].max())
    for j, (classe, cor) in enumerate([(0, COR_CONTEXTO), (1, COR_DESTAQUE)]):
        subset = df[df["battery_failure"] == classe]
        sns.histplot(data=subset, x=col, kde=True, color=cor, ax=axes[i, j])
        axes[i, j].set_xlim(lim)
        label = "No Failure" if classe == 0 else "Failure"
        axes[i, j].set_title(f"{col} — {label}")

fig.suptitle("Distribuição das Variáveis Quantitativas: EV Battery (por classe, em painéis separados)", fontsize=14, y=1.001)
plt.tight_layout()
plt.show()

**Comparação dos grupos na versão analisada:** a saúde média da bateria é aproximadamente 82,67% no grupo sem falha e 71,55% no grupo com falha. A resistência interna média é aproximadamente 0,432 e 0,633, respectivamente. O grupo com falha também apresenta maiores médias de idade, quilometragem e ciclos. Essas estatísticas descrevem associação; o formato das distribuições e sua sobreposição devem ser avaliados nos histogramas, sem supor que médias diferentes tornam as classes perfeitamente separáveis.

#### Testes de normalidade

Vamos investigar `battery_health_percent` com Shapiro-Wilk e QQ-plot.

In [ ]:
def shapiro_resumo(serie):
    valores = serie.replace([np.inf, -np.inf], np.nan).dropna()
    if len(valores) < 3 or valores.nunique() < 2:
        return {"n_teste": len(valores), "estatistica": np.nan,
                "p_valor": np.nan, "rejeita_normalidade": None}
    amostra = valores.sample(n=min(len(valores), 5000), random_state=42)
    estatistica, p_valor = stats.shapiro(amostra)
    return {"n_teste": len(amostra), "estatistica": estatistica,
            "p_valor": p_valor, "rejeita_normalidade": bool(p_valor < 0.05)}

shapiro_antes = shapiro_resumo(df["battery_health_percent"])
print("Shapiro-Wilk — saúde da bateria (antes):", shapiro_antes)
fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(df["battery_health_percent"].dropna(), dist="norm", plot=ax)
ax.get_lines()[0].set_markerfacecolor(COR_NEUTRA)
ax.get_lines()[0].set_markeredgecolor(COR_NEUTRA)
ax.get_lines()[1].set_color(COR_ALERTA)
ax.set_title("QQ-plot — saúde da bateria versus normal")
plt.tight_layout()
plt.show()

**Como ler:** pontos próximos da reta no QQ-plot são compatíveis com normalidade; desvios sistemáticos sugerem diferenças. No Shapiro-Wilk, p < 0,05 é evidência contra normalidade; p ≥ 0,05 não a comprova. Usamos amostra reprodutível de até 5.000 observações, pois o p-valor do Shapiro-Wilk pode ser impreciso acima desse tamanho. Considere também forma, tamanho da amostra e relevância prática.

In [ ]:
resultados_shapiro = [{"variavel": col, **shapiro_resumo(df[col])}
                       for col in VARS_QUANT]
pd.DataFrame(resultados_shapiro).set_index("variavel")

**ECDF — distribuição acumulada**

Comparar a saúde da bateria nos grupos com e sem falha, sem depender da escolha dos intervalos de um histograma.

In [ ]:
plt.figure(figsize=(8, 5))
sns.ecdfplot(data=df, x="battery_health_percent",
             hue="battery_failure",
             palette={0: COR_CONTEXTO, 1: COR_DESTAQUE})
plt.title("ECDF — Battery Health % por Presença de Falha na Bateria")
plt.xlabel("Battery Health (%)")
plt.ylabel("Proporção acumulada")
plt.tight_layout()
plt.show()

Em cada valor do eixo X, a curva mostra a fração de registros do grupo cuja saúde da bateria é menor ou igual àquele valor. Diferenças entre as curvas indicam diferenças entre distribuições, sem provar uma relação causal.

### Métricas de Tendêncial Central

In [ ]:
media_bh = df["battery_health_percent"].mean()
mediana_bh = df["battery_health_percent"].median()

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="battery_health_percent", kde=True, color=COR_NEUTRA)
plt.axvline(media_bh, color=COR_ALERTA, linestyle="--", linewidth=2, label=f"Média = {media_bh:.1f}")
plt.axvline(mediana_bh, color="black", linestyle="-", linewidth=2, label=f"Mediana = {mediana_bh:.1f}")
plt.title("Battery Health %: Distribuição com Média e Mediana")
plt.legend()
plt.tight_layout()
plt.show()

### Dispersão

In [ ]:
dispersao = df[VARS_QUANT].agg(["var", "std"]).T
dispersao["IQR"] = df[VARS_QUANT].quantile(0.75) - df[VARS_QUANT].quantile(0.25)
print(dispersao.round(2))

# Uma linha por variável evita uma figura excessivamente larga.
fig, axes = plt.subplots(len(VARS_QUANT), 2, figsize=(10, 3 * len(VARS_QUANT)))
for i, col in enumerate(VARS_QUANT):
    sns.boxplot(data=df, x="battery_failure", y=col,
                color=COR_NEUTRA, ax=axes[i, 0])
    sns.violinplot(data=df, x="battery_failure", y=col,
                   color=COR_NEUTRA, ax=axes[i, 1])
    axes[i, 0].set_title(f"Boxplot — {col}")
    axes[i, 1].set_title(f"Violino — {col}")
plt.tight_layout()
plt.show()

**Discussão:** quais variáveis têm medianas e dispersões diferentes entre os grupos sem falha (0) e com falha (1)? Os boxplots e violinos complementam os histogramas.

## 4. Tratamento de Dados Ausentes e Outliers

Usaremos a inspeção anterior para tratar ausentes nas variáveis selecionadas, manter os demais campos e avaliar outliers sem removê-los automaticamente.

### Detecção de Outliers por IQR

A regra clássica marca como outlier qualquer ponto fora de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.

In [ ]:
def limites_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

resumo_outliers = {}
for col in VARS_QUANT:
    lim_inf, lim_sup = limites_iqr(df[col])
    n_outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    resumo_outliers[col] = {
        "limite_inferior": round(lim_inf, 2),
        "limite_superior": round(lim_sup, 2),
        "n_outliers": n_outliers,
    }

pd.DataFrame(resumo_outliers).T

### Visualizando os Outliers

In [ ]:
import math
ncols = 3
nrows = math.ceil(len(VARS_QUANT) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3 * nrows))
axes = axes.ravel()
for ax, col in zip(axes, VARS_QUANT):
    sns.boxplot(data=df, y=col, color=COR_NEUTRA,
                flierprops=dict(markerfacecolor=COR_ALERTA), ax=ax)
    ax.set_title(col)
for ax in axes[len(VARS_QUANT):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Discussão:** outliers de quilometragem, resistência ou temperatura podem ser valores raros legítimos ou erros. A regra do IQR sinaliza pontos para investigação, não estabelece limites físicos. Não excluímos registros apenas por ultrapassarem esses limites.

### Tratamento dos dados ausentes

Para preservar a opção feita na EDA original, removemos linhas com ausentes nas 13 quantitativas e nas 8 qualitativas selecionadas. Isso **não garante que as demais colunas fiquem completas**. Com muitas variáveis, a perda acumulada de linhas pode ser grande: calculamos quantidade e percentual removidos e comparamos a proporção de falhas antes/depois. Sem conhecer o mecanismo de ausência, não podemos afirmar ausência de viés. Imputação seria uma alternativa a avaliar, não uma mudança aplicada silenciosamente.

In [ ]:
df_antes = df.copy(deep=True)
linhas_antes = len(df_antes)
df = df.dropna(subset=VARS_QUANT + VARS_QUALI).reset_index(drop=True)
linhas_depois = len(df)
if df.empty:
    raise ValueError("O tratamento removeu todas as linhas; revise a estratégia de ausentes.")
removidas = linhas_antes - linhas_depois
print(f"Antes: {linhas_antes} | Depois: {linhas_depois} | "
      f"Removidas: {removidas} ({removidas / linhas_antes:.2%})")
comparacao_classes = pd.DataFrame({
    "antes": df_antes["battery_failure"].value_counts(normalize=True),
    "depois": df["battery_failure"].value_counts(normalize=True),
}).reindex([0, 1], fill_value=0).fillna(0)
print("Proporção de classes antes/depois:")
print(comparacao_classes.round(4))
print("Ausentes remanescentes em outras colunas:", int(df.isna().sum().sum()))

### Tratamento dos outliers

Mantemos os valores originais. Criamos `odometer_km_log = log(1 + odometer_km)` como alternativa de escala, preservando `odometer_km`. A transformação exige quilometragem não negativa e não garante normalidade nem redução da assimetria.

In [ ]:
if (df["odometer_km"] < 0).any():
    raise ValueError("Há quilometragem negativa; investigar antes de aplicar log1p.")

df["odometer_km_log"] = np.log1p(df["odometer_km"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(data=df, x="odometer_km", kde=True, color=COR_NEUTRA, ax=axes[0])
axes[0].set_title(f"odometer_km original (skew={df['odometer_km'].skew():.2f})")

sns.histplot(data=df, x="odometer_km_log", kde=True, color=COR_NEUTRA, ax=axes[1])
axes[1].set_title(f"log(1 + odometer_km) (skew={df['odometer_km_log'].skew():.2f})")

plt.tight_layout()
plt.show()

**Discussão:** compare a assimetria da quilometragem antes e depois da transformação. A coluna logarítmica não substitui a original nem altera a seleção principal das próximas análises.

## 5. Estatística Descritiva pt 2

Fechando o ciclo iterativo: tratamos os ausentes e decidimos manter os outliers. Agora rodamos a mesma bateria de estatísticas da Seção 3 para comparar diretamente com o "antes".

In [ ]:
desc_depois = df[VARS_QUANT].describe().T
desc_depois["skewness"] = df[VARS_QUANT].skew()
desc_depois["kurtosis"] = df[VARS_QUANT].kurtosis()

(
    desc_depois[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    
    
)

### Comparando "antes"x "depois""



In [ ]:
comparacao = pd.DataFrame({
    "skewness_antes": desc["skewness"],
    "skewness_depois": desc_depois["skewness"],
    "kurtosis_antes": desc["kurtosis"],
    "kurtosis_depois": desc_depois["kurtosis"],
})
comparacao["delta_skewness"] = (comparacao["skewness_depois"] - comparacao["skewness_antes"]).round(3)
comparacao["delta_kurtosis"] = (comparacao["kurtosis_depois"] - comparacao["kurtosis_antes"]).round(3)
comparacao.round(3)

**Discussão:** avalie as mudanças de distribuição juntamente com o percentual de linhas removidas. Pequenas alterações nas estatísticas não comprovam que a seleção foi isenta de viés.

**A saúde da bateria continua apresentando evidências contra normalidade?**

Aplicamos o mesmo procedimento de Shapiro-Wilk após o tratamento.

In [ ]:
shapiro_depois = shapiro_resumo(df["battery_health_percent"])
print("Shapiro-Wilk — saúde da bateria (antes):", shapiro_antes)
print("Shapiro-Wilk — saúde da bateria (depois):", shapiro_depois)

**Comparação antes/depois**

Sobrepor as distribuições de `battery_health_percent` e `internal_resistance` antes e depois da remoção de ausentes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, ["battery_health_percent", "internal_resistance"]):
    sns.kdeplot(df_antes[col].dropna(), color=COR_CONTEXTO, fill=True, label="Antes", ax=ax)
    sns.kdeplot(df[col].dropna(), color=COR_DESTAQUE, fill=True, label="Depois", ax=ax)
    ax.set_title(col)
    ax.legend()
fig.suptitle("Distribuições antes e depois do tratamento")
plt.tight_layout()
plt.show()

**Discussão:** observe deslocamento, dispersão e caudas das curvas antes/depois. Use densidade para comparar a forma apesar dos tamanhos diferentes das amostras.

## 6. Princípio Relacional

### Análise de Variáveis Qualitativas

Agora focaremos na avaliação de frequência e desbalanceamento das variáveis qualitativas.




#### Frequências absolutas e relativas

In [ ]:
for col in VARS_QUALI:
    freq_abs = df[col].value_counts()
    freq_rel = df[col].value_counts(normalize=True).round(3)
    print(f"\\n--- {col} ---")
    print(pd.concat([freq_abs.rename("absoluta"), freq_rel.rename("relativa")], axis=1))

#### Visualizando as frequências

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=True)
axes = axes.ravel()

for i, col in enumerate(VARS_QUALI):
    df[col].value_counts().sort_index().plot(kind="bar", color=COR_NEUTRA, ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis="x", rotation=0)

fig.suptitle("Frequências das Variáveis Qualitativas", fontsize=14)
plt.tight_layout()
plt.show()

**Pra discutir:** alguma variável qualitativa está bem desbalanceada (uma categoria dominando quase todas as observações)? Isso pode impactar como vamos interpretar cruzamentos com essa variável mais na frente na análise bivariada.

#### Desbalanceamento entre classes

- Razão entre maior e menor contagem observada: quanto maior, maior o desbalanceamento.
- Entropia normalizada: próximo de 1 indica proporções mais uniformes entre as categorias observadas; 0 indica uma única categoria.

Inspecionamos as qualitativas selecionadas, incluindo `battery_failure`, e contamos explicitamente as duas classes esperadas do alvo.

In [ ]:
from scipy.stats import entropy

def metrica_desbalanceamento(serie):
    contagens = serie.value_counts()
    contagens = contagens[contagens > 0]
    n_classes = len(contagens)
    if n_classes == 0:
        return pd.Series({"n_classes": 0, "razao_desbalanceamento": np.nan,
                          "entropia_normalizada": np.nan})
    props = contagens / contagens.sum()
    ent = entropy(props, base=2) / np.log2(n_classes) if n_classes > 1 else 0.0
    razao = contagens.max() / contagens.min() if n_classes > 1 else np.inf
    return pd.Series({"n_classes": n_classes, "razao_desbalanceamento": razao,
                      "entropia_normalizada": ent})

print("Contagens das classes esperadas do alvo:")
print(df["battery_failure"].value_counts().reindex([0, 1], fill_value=0))
pd.DataFrame({col: metrica_desbalanceamento(df[col]) for col in VARS_QUALI}).T

**Discussão:** quais variáveis apresentam maiores desequilíbrios? Confira também se ambas as classes do alvo estão presentes antes de comparar grupos.

### Análise Bivariada

Investigar associações entre variáveis e diferenças entre registros com e sem falha de bateria.

#### Quantitativa × quantitativa — correlação

**Pearson** mede associação linear e é sensível a outliers. Seu cálculo não exige normalidade; os pressupostos de testes e intervalos de confiança são uma questão distinta. **Spearman** e **Kendall** medem associação por postos e ajudam a investigar tendências monotônicas. Comparamos as três medidas sem escolher uma apenas pelo resultado do teste de normalidade.

In [ ]:
corr_pearson = df[VARS_QUANT].corr(method="pearson")
corr_spearman = df[VARS_QUANT].corr(method="spearman")
corr_kendall = df[VARS_QUANT].corr(method="kendall")

fig, ax = plt.subplots(1,3,figsize=(24, 7),sharey=True)
sns.heatmap(corr_pearson, annot=True, annot_kws={"size": 6}, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[0])
ax[0].set_title("Matriz de Correlação — Pearson")
sns.heatmap(corr_spearman, annot=True, annot_kws={"size": 6}, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[1])
ax[1].set_title("Matriz de Correlação — Spearman")
sns.heatmap(corr_kendall, annot=True, annot_kws={"size": 6}, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[2])
ax[2].set_title("Matriz de Correlação — Kendall")
plt.tight_layout()
plt.show()

Como os dados do triângulo superior e inferior são simétricos, poderíamos plotar a matriz de correlação da seguinte forma.

In [ ]:
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))

plt.figure(figsize=(11, 9))
sns.heatmap(corr_spearman, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", square=True, vmin=-1, vmax=1,)
plt.grid(False)

plt.tight_layout()
plt.show()

**Discussão:** diferenças entre Pearson e Spearman podem refletir não linearidade, outliers ou outros aspectos da distribuição. Inspecione o gráfico de dispersão antes de concluir.

#### Dispersão entre variáveis quantitativas

Investigar `cycle_count` × `battery_health_percent`: o número de ciclos se associa à saúde da bateria? Comparar as classes de falha sem interpretar associação como causa.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="cycle_count", y="battery_health_percent",
                hue="falha_label", palette={"Sem falha": COR_CONTEXTO,
                "Com falha": COR_DESTAQUE}, alpha=0.4, s=15)
plt.title("Ciclos × saúde da bateria")
plt.xlabel("Ciclos")
plt.ylabel("Saúde da bateria (%)")
plt.tight_layout()
plt.show()

**Discussão:** há tendência de redução da saúde com mais ciclos? Existe sobreposição entre os grupos? Confirme no gráfico e nas correlações.

#### Quantitativa × qualitativa — comparação de dois grupos

Comparamos cada quantitativa entre `battery_failure = 0` e `1`. O teste t de Welch avalia diferença de médias, enquanto Mann-Whitney avalia distribuições/postos; não são testes da mesma hipótese e não precisam concordar. A normalidade deve ser discutida dentro dos grupos, e amostras grandes podem tornar o teste t robusto. Aqui os testes são exploratórios; muitos testes simultâneos aumentam o risco de falsos positivos.

In [ ]:
resultados_comparacao = []
for col in VARS_QUANT:
    grupo_sem = df.loc[df["battery_failure"] == 0, col].dropna()
    grupo_com = df.loc[df["battery_failure"] == 1, col].dropna()
    if min(len(grupo_sem), len(grupo_com)) < 2:
        print(f"{col}: grupos insuficientes; comparação não realizada.")
        continue
    stat_t, p_t = stats.ttest_ind(grupo_sem, grupo_com, equal_var=False)
    stat_u, p_u = stats.mannwhitneyu(grupo_sem, grupo_com, alternative="two-sided")
    resultados_comparacao.append({"variavel": col, "n_sem_falha": len(grupo_sem),
        "n_com_falha": len(grupo_com), "media_sem_falha": grupo_sem.mean(),
        "media_com_falha": grupo_com.mean(), "p_valor_welch": p_t,
        "p_valor_mannwhitney": p_u})
pd.DataFrame(resultados_comparacao)

**Discussão:** compare os resultados de Welch e Mann-Whitney considerando as hipóteses distintas. Significância não informa o tamanho ou a importância prática da diferença. Os p-valores abaixo não receberam correção por múltiplas comparações.

**Três ou mais grupos:** comparar saúde da bateria entre categorias de `battery_chemistry`, com ANOVA e Kruskal-Wallis como análises exploratórias.

In [ ]:
grupos_quimica = [g["battery_health_percent"].dropna().to_numpy()
                   for _, g in df.groupby("battery_chemistry", observed=True)
                   if g["battery_health_percent"].notna().sum() >= 2]
if len(grupos_quimica) >= 2 and np.unique(np.concatenate(grupos_quimica)).size > 1:
    stat_anova, p_anova = stats.f_oneway(*grupos_quimica)
    stat_kw, p_kw = stats.kruskal(*grupos_quimica)
    print(f"ANOVA — saúde por química: estatística={stat_anova:.3f}, p={p_anova:.5g}")
    print(f"Kruskal-Wallis — saúde por química: estatística={stat_kw:.3f}, p={p_kw:.5g}")
else:
    print("Não há grupos ou variação suficientes para os testes.")
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="battery_chemistry", y="battery_health_percent", color=COR_NEUTRA)
plt.title("Saúde da bateria por química")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**Discussão:** há evidência de diferença entre grupos de química da bateria? ANOVA e Kruskal-Wallis não identificam sozinhos quais pares diferem. Inspecione tamanhos, variâncias e formas das distribuições.

#### Qualitativa × qualitativa

In [ ]:
def analisar_associacao(atributo):
    tabela = pd.crosstab(df[atributo], df["battery_failure"])
    if min(tabela.shape) < 2:
        print(f"{atributo}: categorias insuficientes para qui-quadrado.")
        return
    chi2, p_valor, gl, esperado = stats.chi2_contingency(tabela)
    print(f"{atributo}: p={p_valor:.5g}; menor frequência esperada={esperado.min():.2f}")
    if (esperado < 5).any():
        print("Atenção: há frequências esperadas < 5; a aproximação pode ser inadequada.")
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(tabela, annot=True, fmt="d", cmap="magma", ax=ax)
    ax.set_title(f"{atributo} × falha da bateria (p={p_valor:.4g})")
    plt.tight_layout()
    plt.show()

analisar_associacao("battery_chemistry")

**Discussão:** existe associação entre `battery_chemistry` e `battery_failure`? Verifique as frequências esperadas antes de interpretar a aproximação do qui-quadrado. Associação não prova causalidade.

#### Comparação entre subgrupos

Vamos comparar a proporção de falhas por tipo de veículo e faixa de idade. São subgrupos operacionais: este dataset não fornece aqui uma análise de justiça algorítmica sobre grupos humanos protegidos. Diferenças podem refletir uso, composição da amostra ou outros fatores não controlados.

In [ ]:
df["faixa_idade_veiculo"] = pd.cut(
    df["vehicle_age_years"], bins=[-np.inf, 3, 6, 10, np.inf],
    labels=["Até 3 anos", ">3 até 6 anos", ">6 até 10 anos", ">10 anos"], right=True)
for atributo in ["vehicle_type", "faixa_idade_veiculo"]:
    print(atributo)
    print(metrica_desbalanceamento(df[atributo]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, atributo in zip(axes, ["vehicle_type", "faixa_idade_veiculo"]):
    resumo = df.groupby(atributo, observed=True)["battery_failure"].agg(
        registros="size", proporcao_falhas="mean")
    print(atributo)
    print(resumo)
    resumo["proporcao_falhas"].plot(kind="bar", color=COR_DESTAQUE, ax=ax)
    ax.set_title(f"Proporção de falhas por {atributo}")
    ax.set_ylabel("Proporção de falhas")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

**Discussão:** como a proporção de falhas varia por tipo de veículo e faixa de idade? Considere as contagens de cada grupo junto às proporções e evite atribuir causalidade.

In [ ]:
for atributo in ["vehicle_type", "faixa_idade_veiculo"]:
    analisar_associacao(atributo)

**Discussão:** os testes indicam associação entre os subgrupos operacionais e falha? Confira frequências esperadas e possíveis confundidores. A análise não demonstra causa nem justiça de um modelo.

#### Outliers bivariados

In [ ]:
sns.jointplot(data=df, x="cycle_count", y="internal_resistance", hue="falha_label",
              palette={"Sem falha": COR_CONTEXTO, "Com falha": COR_DESTAQUE},
              alpha=0.4, height=6)
plt.suptitle("Ciclos × resistência interna: combinações incomuns?", y=1.02)
plt.show()

**Discussão:** existem combinações incomuns de `cycle_count` e `internal_resistance` que passariam despercebidas na análise de cada variável isolada?

#### Análise Multivariada

##### Matriz de Correlação

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df[VARS_QUANT].corr(method="spearman"), annot=True, fmt=".2f", annot_kws={"size": 7}, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de Correlação (Spearman) — Variáveis Quantitativas")
plt.tight_layout()
plt.show()

**Interpretação aproximada da magnitude:** |r| < 0,3: fraca; 0,3 ≤ |r| < 0,7: moderada; |r| ≥ 0,7: forte. São heurísticas dependentes do domínio. Observe o sinal e não interprete correlação como causalidade.

##### Correlograma

Para tornar a execução local mais leve, usamos até 1.000 registros com amostragem reprodutível e seis variáveis. As matrizes anteriores utilizam todas as linhas tratadas e as 13 variáveis selecionadas. Os coeficientes deste gráfico são Pearson na amostra e podem diferir das matrizes anteriores.

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

def circulo_correlacao(x, y, **kwargs):
    pares = pd.DataFrame({"x": x, "y": y}).dropna()
    r = pares["x"].corr(pares["y"]) if len(pares) > 1 else np.nan
    r = float(r) if pd.notna(r) else 0.0
    ax = plt.gca()
    ax.scatter(
        [0.5], [0.5],
        s=abs(r) * 4000,          # tamanho proporcional à força da correlação
        c=[r], cmap='coolwarm', vmin=-1, vmax=1,
        transform=ax.transAxes, edgecolor="white", linewidth=0.5,
    )
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes,
                ha="center", va="center", fontsize=10,
                color="white" if abs(r) > 0.5 else "black")
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(False)

VARS_PAIRGRID = ["vehicle_age_years", "odometer_km", "cycle_count",
                 "battery_health_percent", "internal_resistance", "aging_score"]
amostra_pairgrid = df[VARS_PAIRGRID].sample(n=min(len(df), 1000), random_state=42)
g = sns.PairGrid(amostra_pairgrid, diag_sharey=False)
g.map_lower(sns.scatterplot, color=COR_NEUTRA, alpha=0.6, s=25)
g.map_diag(sns.histplot, color=COR_NEUTRA, kde=True)
g.map_upper(circulo_correlacao)

# barra de cor única, compartilhada (cada subplot superior tem seu próprio scatter)
sm = ScalarMappable(norm=Normalize(-1, 1), cmap='coolwarm')
sm.set_array([])
g.fig.colorbar(sm, ax=g.axes, shrink=0.6, label="Correlação (r)")

g.fig.suptitle("Correlograma — amostra de até 1.000 registros", y=1.02, fontsize=15)

## Síntese e declaração de uso de IA

Resultados verificados com o CSV obtido da fonte indicada neste notebook:

1. **Estrutura:** 20.000 registros, 70 colunas e nenhuma linha inteiramente duplicada. Há ausentes em 67 colunas; identificadores `vehicle_id` e `battery_serial` e o alvo estão completos.
2. **Efeito do tratamento:** a remoção de ausentes nas variáveis selecionadas deixa 8.792 registros, eliminando 11.208 (56,04%). A proporção de falhas passa de 6,92% para aproximadamente 6,54%. Essa redução deve ser considerada nas conclusões; não é evidência de ausência de viés.
3. **Diferenças entre grupos:** nos dados originais, a saúde média é 82,67% sem falha e 71,55% com falha; a resistência interna média é 0,432 e 0,633, respectivamente. Há sobreposição entre distribuições e essas diferenças não demonstram causalidade.
4. **Associação e redundância:** nos dados tratados, Spearman entre saúde da bateria e resistência interna é aproximadamente −0,914; entre saúde e ciclos, −0,496. `battery_health_percent` e `state_of_health` têm correlação de aproximadamente 0,989, indicando possível redundância.
5. **Escalas e limitações:** o logaritmo da quilometragem muda sua assimetria de aproximadamente 0,835 para −0,782, sem garantir normalidade. Mantivemos outliers. O alvo é desbalanceado e os testes exploratórios não receberam correção por múltiplas comparações.

Os números são uma referência da versão utilizada na verificação; se os dados mudarem, execute novamente e atualize a síntese.

**Declaração desta revisão:** ChatGPT foi utilizado para revisar código e explicações, substituir referências restantes do exemplo de doenças cardíacas, adicionar verificações e exportação para Streamlit e limitar gráficos de custo elevado. A versão revisada foi executada com o dataset indicado, e suas saídas fundamentaram a síntese acima. Foram preservados a seleção principal de variáveis e o tratamento de ausentes do aluno, com explicitação de suas limitações. As alterações estão aplicadas nesta versão; o aluno deve revisá-las e complementar a declaração com o que efetivamente aceitou, modificou ou rejeitou e por quê. Não se presume que ele já tenha validado as alterações.

## Exportação para o Streamlit
O CSV tratado preserva as 70 colunas originais e as derivadas. O tratamento garante completude apenas nas variáveis selecionadas. `data/raw/` fica fora do Git conforme o modelo; versione `data/clean/ev_battery_clean.csv`. O app deve ler o CSV tratado, sem baixar dados a cada interação.


In [ ]:
ARQUIVO_CLEAN = PASTA_CLEAN / "ev_battery_clean.csv"
df.to_csv(ARQUIVO_CLEAN, index=False)
print(f"Original: {ARQUIVO_RAW} ({len(df_original)} registros)")
print(f"Tratado: {ARQUIVO_CLEAN} ({len(df)} registros, {df.shape[1]} colunas)")
print("Variáveis selecionadas completas:", not df[VARS_QUANT + VARS_QUALI].isna().any().any())
